# 02 · Generate DFT labels for fine-tuning

## Objective

This notebook reads 10 structures from an existing extxyz file, calculates their energy, force, stress, and charge labels with GPAW and Chargemol. For finetuning, we also need isolated-atom energies with identical settings which provide a reference for model fine-tuning.

This notebook shows labelling as a separate step. Notebook 01 can also include labelling in its generation workflow. Here, use the geometry from the bundled sample file.

Note that this notebook stores calculation results in a configured Jobstore which is used by Notebook 03 to retrieve the stored labels and isolated-atom references for fine-tuning.

## Prerequisites

Before executing the notebook, make sure the following dependencies are met,

1. Install the notebook group and GPAW dependencies:

   ```bash
   uv sync --group notebooks --group gpaw
   ```

2. Install the GPAW PAW datasets and Chargemol. Set `gpaw_env_dir`, `gpaw_setups_dir`, and `chargemol_dir` in `crystal.toml`. Follow the [installation instructions](../README.md#3-install-gpaw-paw-datasets-and-chargemol) in the main README. A compatible `mpirun` must be available.
3. Start MongoDB. Check the connection settings in `.jobstore/local/jobstore.yaml`.


## 1. Import the labelling tools

Import the structure reader, calculation makers, settings, and result types. Use the connection helper to access the configured JobStore.

In [ ]:
from pathlib import Path

from ase import Atoms
from ase.io import read
from jobflow.core.flow import Flow
from jobflow.core.reference import OutputReference
from jobflow.managers.local import run_locally
from monty.serialization import dumpfn, loadfn
from pymatgen.io.ase import AseAtomsAdaptor

from crystal import CRYSTAL_ROOT, LOCAL_JOBSTORE_YAML
from crystal.calculators.gpaw.isolated_atom_maker import GpawIsolatedAtomMaker
from crystal.calculators.gpaw.scf_chargemol_maker import GpawScfChargemolMaker
from crystal.helpers.fireworks.connections import connect_to_jobstore
from crystal.schemas.gpaw import (
    GpawChargemolSimulationSettings,
    GpawConvergenceSettings,
    GpawIsolatedAtomCalculatorSettings,
    GpawIsolatedAtomConvergenceSettings,
    GpawIsolatedAtomSimulationSettings,
    GpawScfChargemolSimulationSettings,
    GpawScfChargemolTaskDocument,
    GpawScfSimulationSettings,
    GpawSettings,
)
from crystal.schemas.gpaw_isolated_atom_reference import (
    GpawIsolatedAtomReferenceCollection,
)


## 2. Set paths, constants, and the run flag

Select the input extxyz file and request 10 structures. Use `RUN_CALCULATIONS=True` to start a new calculation. Set it to `False` to inspect a completed campaign without repeating the calculations.

The campaign file, `results/label-jobs.json`, records the label job UUIDs, the isolated-atom collection UUID, and the JobStore configuration path. Notebook 03 uses this file to find the correct results. Keep the file and the JobStore records.

In [ ]:
RUN_CALCULATIONS = True
NUM_STRUCTURES = 10
# These are only for demo purposes for faster execution
CUTOFF_EV = 200.0
KPOINT_DENSITY = 1.0

DATA_DIR = CRYSTAL_ROOT / "examples/results"
STRUCTURES_PATH = CRYSTAL_ROOT / "examples/data/mock_structures.extxyz"
CAMPAIGN_PATH = DATA_DIR / "label-jobs.json"
RUN_DIRECTORY = DATA_DIR / "label-runs"
JOBSTORE_CONFIG = LOCAL_JOBSTORE_YAML


## 3. Read 10 input structures

Read the first 10 frames from the extxyz file. Check the count before you start calculations. Copy only the atom symbols, positions, cell, and periodic boundary settings. This removes the old calculation labels and metadata from the new inputs.

Change `STRUCTURES_PATH` to use another extxyz file. This cell does not run DFT calculations.

In [ ]:
input_frames = read(STRUCTURES_PATH, index=f":{NUM_STRUCTURES}")
if len(input_frames) != NUM_STRUCTURES:
    msg = f"Expected {NUM_STRUCTURES} structures in {STRUCTURES_PATH}; found {len(input_frames)}."
    raise ValueError(msg)

structures = [
    AseAtomsAdaptor.get_structure(
        Atoms(
            symbols=frame.get_chemical_symbols(),
            positions=frame.positions,
            cell=frame.cell,
            pbc=frame.pbc,
        )
    )
    for frame in input_frames
]
print(f"Structures to label: {len(structures)}")
print(f"Required elements: {sorted({element for structure in structures for element in structure.symbol_set})} for isolated atom records")


## 4. Configure GPAW and Chargemol

Uses a 200 eV cutoff and a k-point density of 1.0 and sets a loose SCF stopping criteria: energy 0.01 eV per valence electron, density 0.1, force change 1.0 eV/Å, and eigenstates 0.001. These settings reduce the execution time for this demonstration. They do not ensure convergence or accurate labels. Check convergence before scientific use.

Chargemol calculates DDEC6 charges after each GPAW calculation. The isolated-atom calculations use the same cutoff and energy, density, and eigenstate tolerances. They use Gamma-only sampling and no force convergence criterion.

In [ ]:
scf_convergence = GpawConvergenceSettings(
    energy=1e-2,
    density=0.1,
    forces=1.0,
    eigenstates=1e-3,
)
label_maker = GpawScfChargemolMaker(
    simulation_settings=GpawScfChargemolSimulationSettings(
        gpaw_scf_simulation_settings=GpawScfSimulationSettings(
            gpaw_settings=GpawSettings(
                cutoff_ev=CUTOFF_EV,
                kpoint_density=KPOINT_DENSITY,
                convergence=scf_convergence,
            ),
        ),
        gpaw_chargemol_simulation_settings=GpawChargemolSimulationSettings(),
    ),
)
atom_settings = GpawIsolatedAtomSimulationSettings(
    calculator_settings=GpawIsolatedAtomCalculatorSettings(
        cutoff_ev=CUTOFF_EV,
        convergence=GpawIsolatedAtomConvergenceSettings(
            energy=scf_convergence.energy,
            density=scf_convergence.density,
            eigenstates=scf_convergence.eigenstates,
        ),
    ),
)


## 5. Run GPAW labelling locally

This step 
1. Builds one GPAW/Chargemol job for each structure, and adds the isolated-atom workflow for all required elements. 

2. Saves the campaign file before execution. This lets you inspect completed outputs if a later job fails. Run the workflow with the configured JobStore. Each job writes its calculation files to a separate directory under `results/label-runs/`.

This cell executes in approximately **2 minutes**. The actual time depends on the computer, structure sizes, and SCF convergence. Wait for the cell to finish before you inspect the labels. An unsuccessful job can stop execution with an error.

Run this cell only once per campaign. Running it again starts new jobs and replaces `label-jobs.json`. It does not delete earlier JobStore records.

**FireWorks execution:** To submit the same workflow, configure a `JobManager` with the target LaunchPad, worker, and JobStore. Use `manager.add_to_launchpad(job_or_flow=flow)` instead of `run_locally(...)`. Save the campaign with that JobStore configuration path. Start the workers and wait for all label and isolated-atom jobs to finish. Do not also run the workflow locally.

In [ ]:
if RUN_CALCULATIONS:
    RUN_DIRECTORY.mkdir(parents=True, exist_ok=True)
    jobstore = connect_to_jobstore(JOBSTORE_CONFIG)
    label_jobs = [label_maker.make(structure=structure) for structure in structures]
    atom_flow = GpawIsolatedAtomMaker(simulation_settings=atom_settings).make(
        elements=frozenset(
            element for structure in structures for element in structure.symbol_set
        )
    )
    flow = Flow(
        jobs=[*label_jobs, atom_flow],
        name="Example labels and isolated atoms",
    )
    dumpfn(
        {
            "labels": [job.uuid for job in label_jobs],
            "atoms": atom_flow.output.uuid,
            "jobstore_config": JOBSTORE_CONFIG.as_posix(),
        },
        CAMPAIGN_PATH,
    )
    run_locally(
        flow,
        store=jobstore,
        root_dir=RUN_DIRECTORY,
        create_folders=True,
        log=True,
        ensure_success=True,
    )
else:
    print("Calculations disabled. Retrieve the saved campaign in the next cell.")


## 6. Retrieve and inspect the labels

This step reads and verifies the campaign file. 
1. Connect to its JobStore and retrieve the exact saved outputs with `OutputReference.resolve(...)`. This decodes stored metadata before result validation. Check that all 10 labels and the isolated-atom reference are available. Print each structure's energy and label counts.

2. You can repeat this cell without repeating the calculations. After a kernel restart, run the imports and constants cells, set `RUN_CALCULATIONS=False`, and run this cell. Keep the campaign path unchanged. If an output is missing, inspect the failed job before you continue.

In [ ]:
if not CAMPAIGN_PATH.is_file():
    msg = "No campaign file. Run the labelling workflow first."
    raise FileNotFoundError(msg)
campaign = loadfn(CAMPAIGN_PATH)
if len(campaign["labels"]) != NUM_STRUCTURES:
    msg = f"The saved campaign must contain {NUM_STRUCTURES} label jobs."
    raise ValueError(msg)
jobstore = connect_to_jobstore(Path(campaign["jobstore_config"]))
label_documents = [
    GpawScfChargemolTaskDocument.model_validate(
        OutputReference(uuid).resolve(jobstore)
    )
    for uuid in campaign["labels"]
]
atom_collection = GpawIsolatedAtomReferenceCollection.model_validate(
    OutputReference(campaign["atoms"]).resolve(jobstore)
)
if atom_collection.reference is None:
    msg = "Isolated-atom calculations failed. Inspect the stored task documents."
    raise RuntimeError(msg)

for index, document in enumerate(label_documents, start=1):
    properties = document.properties
    if not document.converged or properties is None or properties.charges is None:
        msg = f"Label {index} is incomplete. Inspect the task document: {document.errors}"
        raise RuntimeError(msg)
    print(
        f"{index}: {properties.structure.composition.reduced_formula}; "
        f"energy={properties.energy:.6f} eV; "
        f"force vectors={len(properties.forces)}; "
        f"stress components={len(properties.stress)}; "
        f"charges={len(properties.charges)}"
    )
print("Isolated-atom energies (eV):", atom_collection.reference.energies_by_element)
print("Campaign for Notebook 03:", CAMPAIGN_PATH)


## Next step: fine-tune the model

Open [Notebook 03](03_finetune_mlip_and_evaluation.ipynb) after all jobs finish successfully. It reads `results/label-jobs.json` in a new kernel. The `labels` entries identify the GPAW/Chargemol outputs. The `atoms` entry identifies the isolated-atom collection. The `jobstore_config` entry selects the database connection.

Keep the JobStore records, including isolated-atom registry entries. Notebook 03 uses the stored calculation settings, runtime provenance, and physical reference energies to prepare training. 